<a href="https://colab.research.google.com/github/Sneh-04/Machine_Failure_Prediction/blob/main/Machine_Failure_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔧 Machine Failure Prediction using Sensor Data
**Author:** Kunduru Sneha | [GitHub](https://github.com/Sneh-04) | kundurusneha4@gmail.com

---

## 📌 Problem Statement

Industrial machines fail unexpectedly, causing costly downtime and safety risks. This project builds a **binary classification model** to predict machine failures using real-time sensor readings — enabling **proactive maintenance** before breakdowns occur.

**Target variable:** `fail` → 0 = Normal Operation, 1 = Machine Failure

**Dataset:** [Machine Failure Prediction using Sensor Data](https://www.kaggle.com/datasets/umerrtx/machine-failure-prediction-using-sensor-data) — Kaggle  
**Shape:** 1000 rows × 9 columns (8 sensor features + 1 target)

---

## 📋 Table of Contents
1. [Setup & Imports](#1-setup)
2. [Data Loading & Overview](#2-data-loading)
3. [Exploratory Data Analysis (EDA)](#3-eda)
4. [Preprocessing](#4-preprocessing)
5. [Model Training & Comparison](#5-model-training)
6. [Hyperparameter Tuning](#6-tuning)
7. [Final Model Evaluation](#7-evaluation)
8. [Results Summary](#8-summary)


## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve,
                              ConfusionMatrixDisplay)
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, ExtraTreesClassifier)
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'text.color': 'white',
    'grid.color': '#333',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'figure.dpi': 120
})

PALETTE = ['#00d4ff', '#ff4b6e']
print("✅ Libraries loaded successfully")


## 2. Data Loading & Overview <a id='2-data-loading'></a>

We load the dataset and perform an initial inspection to understand its structure, data types, and quality.

In [ ]:
# Load dataset
# If running on Kaggle: path = '/kaggle/input/machine-failure-prediction-using-sensor-data/data.csv'
# If running locally:
data = pd.read_csv('data/data.csv')

print(f"Dataset Shape: {data.shape}")
print(f"\nFeatures: {list(data.columns)}")
data.head(10)


In [ ]:
# Data types and non-null counts
data.info()


In [ ]:
# Statistical summary
data.describe().round(2)


In [ ]:
# Data quality check
print("=" * 40)
print("  DATA QUALITY REPORT")
print("=" * 40)
print(f"  Total rows       : {len(data)}")
print(f"  Total columns    : {data.shape[1]}")
print(f"  Missing values   : {data.isnull().sum().sum()}")
print(f"  Duplicate rows   : {data.duplicated().sum()}")
print("=" * 40)

# Drop duplicates if any
if data.duplicated().sum() > 0:
    data.drop_duplicates(inplace=True)
    print(f"  ✅ Duplicates removed. New shape: {data.shape}")
else:
    print("  ✅ No duplicates found.")


## 3. Exploratory Data Analysis (EDA) <a id='3-eda'></a>

EDA helps us understand the data distribution, identify patterns, detect outliers, and understand feature relationships — all of which guide our modeling decisions.

### 3.1 Target Class Distribution

Since we're predicting failures, understanding **class balance** is critical. Imbalanced data can cause a model to appear accurate while completely ignoring the minority class (failures).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('#0f0f0f')

counts = data['fail'].value_counts()
labels = ['Normal (0)', 'Failure (1)']
colors = PALETTE

# Bar chart
ax1 = axes[0]
bars = ax1.bar(labels, counts.values, color=colors, edgecolor='white', linewidth=0.8, width=0.5)
ax1.set_title('Class Distribution', fontsize=14, fontweight='bold', color='white', pad=15)
ax1.set_ylabel('Count', fontsize=12)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{val}\n({val/len(data)*100:.1f}%)', ha='center', va='bottom',
             fontsize=11, fontweight='bold', color='white')
ax1.set_ylim(0, max(counts.values) * 1.2)

# Pie chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(counts.values, labels=labels, colors=colors,
                                     autopct='%1.1f%%', startangle=90,
                                     textprops={'color': 'white', 'fontsize': 11},
                                     wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
for at in autotexts:
    at.set_fontweight('bold')
ax2.set_title('Class Proportion', fontsize=14, fontweight='bold', color='white', pad=15)

plt.suptitle('Target Variable: Machine Failure Distribution', fontsize=15,
             fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

imbalance_ratio = counts[0] / counts[1]
print(f"\nClass imbalance ratio (Normal:Failure) = {imbalance_ratio:.2f}:1")
if imbalance_ratio > 3:
    print("⚠️  Significant class imbalance detected — will use class_weight='balanced'")
else:
    print("✅ Mild/no class imbalance — standard training applies")


### 3.2 Feature Distributions

Histograms with KDE curves show the distribution of each sensor reading, split by failure class. **Overlapping distributions** suggest low predictive power; **separated distributions** indicate strong predictors.

In [ ]:
features = [c for c in data.columns if c != 'fail']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.patch.set_facecolor('#0f0f0f')
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    for cls, color, label in zip([0, 1], PALETTE, ['Normal', 'Failure']):
        subset = data[data['fail'] == cls][feat]
        ax.hist(subset, bins=30, alpha=0.55, color=color, label=label,
                density=True, edgecolor='none')
        subset.plot.kde(ax=ax, color=color, linewidth=2)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11, fontweight='bold', color='white')
    ax.set_xlabel('Value', fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

patches = [mpatches.Patch(color=PALETTE[0], label='Normal'),
           mpatches.Patch(color=PALETTE[1], label='Failure')]
fig.legend(handles=patches, loc='lower right', fontsize=11,
           facecolor='#1a1a2e', edgecolor='white', labelcolor='white')

plt.suptitle('Sensor Feature Distributions by Class', fontsize=15,
             fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('sensor_histograms.png', bbox_inches='tight', facecolor='#0f0f0f')
plt.show()


### 3.3 Correlation Heatmap

The correlation matrix reveals **linear relationships** between features. High inter-feature correlations (multicollinearity) can affect some models. Strong correlation with `fail` indicates high-value predictors.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
fig.patch.set_facecolor('#0f0f0f')
ax.set_facecolor('#0f0f0f')

corr = data.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5,
            linecolor='#333', ax=ax,
            annot_kws={'size': 9, 'color': 'white'},
            cbar_kws={'shrink': 0.8})

ax.set_title('Feature Correlation Matrix', fontsize=15, fontweight='bold',
             color='white', pad=20)
ax.tick_params(colors='white', labelsize=9)

plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

# Print top correlations with target
print("\n📊 Feature correlations with 'fail' (target):")
target_corr = corr['fail'].drop('fail').sort_values(key=abs, ascending=False)
for feat, val in target_corr.items():
    bar = '█' * int(abs(val) * 20)
    print(f"  {feat:<25} {val:+.4f}  {bar}")


### 3.4 Boxplots — Outlier Detection

Boxplots show the spread and **outliers** in each feature, broken down by failure status. Outliers in failure cases can be informative — machines often show extreme sensor readings before failure.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.patch.set_facecolor('#0f0f0f')
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    data.boxplot(column=feat, by='fail', ax=ax,
                 patch_artist=True,
                 boxprops=dict(facecolor='#1a1a2e', color='white'),
                 medianprops=dict(color='#00d4ff', linewidth=2),
                 whiskerprops=dict(color='white'),
                 capprops=dict(color='white'),
                 flierprops=dict(marker='o', color='#ff4b6e', alpha=0.5, markersize=4))
    ax.set_title(feat.replace('_', ' ').title(), fontsize=10, fontweight='bold', color='white')
    ax.set_xlabel('Failure (0=Normal, 1=Fail)', fontsize=9)
    ax.set_ylabel('Value', fontsize=9)
    ax.title.set_color('white')

for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots: Feature Distribution by Failure Status', fontsize=15,
             fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('boxplots.png', bbox_inches='tight', facecolor='#0f0f0f')
plt.show()


### 3.5 Pairplot of Key Features

A pairplot of the top correlated features gives a **multi-dimensional view** of feature interactions and class separability.

In [ ]:
# Select top 4 features by correlation with target
top_features = target_corr.abs().nlargest(4).index.tolist() + ['fail']
subset_df = data[top_features].copy()
subset_df['fail'] = subset_df['fail'].map({0: 'Normal', 1: 'Failure'})

# Use light style for pairplot (seaborn requirement)
with plt.style.context('dark_background'):
    g = sns.pairplot(subset_df, hue='fail', palette={'Normal': '#00d4ff', 'Failure': '#ff4b6e'},
                     plot_kws={'alpha': 0.5, 's': 20},
                     diag_kind='kde', corner=False)
    g.figure.suptitle('Pairplot: Top Correlated Features', y=1.02,
                       fontsize=14, fontweight='bold', color='white')
    plt.savefig('pairplot.png', bbox_inches='tight', facecolor='#111')
    plt.show()


## 4. Preprocessing <a id='4-preprocessing'></a>

Before training, we:
1. **Separate features (X) and target (y)**
2. **Split** into train/test sets (80/20 stratified split to preserve class ratios)
3. **Scale** features using `StandardScaler` — important for distance-based and gradient-based models


In [ ]:
X = data.drop(columns=['fail'])
y = data['fail']

print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
print(f"Feature names  : {list(X.columns)}")

# Stratified split preserves class proportions in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain size : {X_train.shape[0]} samples")
print(f"Test size  : {X_test.shape[0]} samples")
print(f"\nTrain class balance:\n{y_train.value_counts().to_string()}")
print(f"\nTest class balance:\n{y_test.value_counts().to_string()}")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\n✅ Data split and scaled successfully")


## 5. Model Training & Comparison <a id='5-model-training'></a>

We train **6 different classifiers** and compare their performance. Using multiple models helps us identify which algorithm suits this dataset best, rather than blindly applying one.

| Model | Why it's included |
|---|---|
| Random Forest | Robust ensemble, handles non-linear patterns |
| Gradient Boosting | Sequential boosting, often best out-of-the-box |
| XGBoost | Optimized GB with regularization |
| AdaBoost | Adaptive boosting, sensitive to outliers |
| Extra Trees | Faster RF variant with more randomness |
| Decision Tree | Baseline tree model; interpretable |


In [ ]:
# class_weight='balanced' corrects for any class imbalance automatically
classifiers = {
    'Random Forest'     : RandomForestClassifier(random_state=42, class_weight='balanced'),
    'Gradient Boosting' : GradientBoostingClassifier(random_state=42),
    'XGBoost'           : XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                                         random_state=42, verbosity=0),
    'AdaBoost'          : AdaBoostClassifier(random_state=42),
    'Extra Trees'       : ExtraTreesClassifier(random_state=42, class_weight='balanced'),
    'Decision Tree'     : DecisionTreeClassifier(random_state=42, class_weight='balanced')
}

results = []

print(f"{'Model':<22} {'Accuracy':>10} {'ROC-AUC':>10} {'F1 (Fail)':>12} {'CV Mean':>10}")
print("-" * 68)

for name, clf in classifiers.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]

    acc    = accuracy_score(y_test, y_pred)
    auc    = roc_auc_score(y_test, y_prob)
    report = classification_report(y_test, y_pred, output_dict=True)
    f1_fail = report['1']['f1-score']
    cv_scores = cross_val_score(clf, X_train_scaled, y_train, cv=5, scoring='roc_auc')

    results.append({
        'Model': name, 'Accuracy': acc, 'ROC-AUC': auc,
        'F1-Failure': f1_fail, 'CV-AUC Mean': cv_scores.mean(),
        'CV-AUC Std': cv_scores.std()
    })
    print(f"{name:<22} {acc:>10.4f} {auc:>10.4f} {f1_fail:>12.4f} {cv_scores.mean():>10.4f}")

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print("\n✅ All models trained and evaluated")


In [ ]:
# Visualise model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#0f0f0f')

metrics = ['Accuracy', 'ROC-AUC', 'F1-Failure']
titles  = ['Accuracy', 'ROC-AUC Score', 'F1 Score (Failure Class)']
colors  = ['#00d4ff', '#a78bfa', '#ff4b6e']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    sorted_df = results_df.sort_values(metric)
    bars = ax.barh(sorted_df['Model'], sorted_df[metric],
                   color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.set_xlim(sorted_df[metric].min() * 0.97, 1.02)
    ax.set_title(title, fontsize=13, fontweight='bold', color='white', pad=12)
    ax.set_xlabel('Score', fontsize=10)
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9, color='white', fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold',
             color='white', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

print("\n🏆 Ranking by ROC-AUC:")
print(results_df[['Model', 'Accuracy', 'ROC-AUC', 'F1-Failure', 'CV-AUC Mean']].to_string(index=False))


## 6. Hyperparameter Tuning <a id='6-tuning'></a>

We apply **GridSearchCV with 5-fold cross-validation** on the top 2 performing models to find their optimal hyperparameters. We optimize for **ROC-AUC** rather than accuracy, since it's more informative for imbalanced failure prediction.


In [ ]:
# Tune top model: Gradient Boosting
print("⏳ Tuning Gradient Boosting (this may take ~1–2 minutes)...")
gb = GradientBoostingClassifier(random_state=42)
gb_params = {
    'n_estimators'  : [100, 200, 300],
    'learning_rate' : [0.01, 0.05, 0.1, 0.2],
    'max_depth'     : [3, 5, 7],
    'subsample'     : [0.8, 1.0]
}
gb_grid = GridSearchCV(gb, gb_params, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0)
gb_grid.fit(X_train_scaled, y_train)

print(f"\n✅ Best GB Parameters : {gb_grid.best_params_}")
print(f"   Best CV ROC-AUC    : {gb_grid.best_score_:.4f}")


In [ ]:
# Tune Random Forest
print("⏳ Tuning Random Forest...")
rf = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_params = {
    'n_estimators' : [100, 200, 300],
    'max_depth'    : [5, 10, None],
    'min_samples_split': [2, 5, 10]
}
rf_grid = GridSearchCV(rf, rf_params, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0)
rf_grid.fit(X_train_scaled, y_train)

print(f"\n✅ Best RF Parameters : {rf_grid.best_params_}")
print(f"   Best CV ROC-AUC    : {rf_grid.best_score_:.4f}")

# Pick the best tuned model
if gb_grid.best_score_ >= rf_grid.best_score_:
    best_model = gb_grid.best_estimator_
    best_model_name = "Gradient Boosting (Tuned)"
    print(f"\n🏆 Best overall model: {best_model_name} (CV AUC = {gb_grid.best_score_:.4f})")
else:
    best_model = rf_grid.best_estimator_
    best_model_name = "Random Forest (Tuned)"
    print(f"\n🏆 Best overall model: {best_model_name} (CV AUC = {rf_grid.best_score_:.4f})")


## 7. Final Model Evaluation <a id='7-evaluation'></a>

We evaluate the best tuned model on the **held-out test set** using:
- **Confusion Matrix** — actual vs predicted classes
- **Classification Report** — precision, recall, F1 per class
- **ROC Curve** — trade-off between sensitivity and specificity
- **Feature Importance** — which sensors drive predictions most


In [ ]:
y_pred_final = best_model.predict(X_test_scaled)
y_prob_final = best_model.predict_proba(X_test_scaled)[:, 1]

final_acc = accuracy_score(y_test, y_pred_final)
final_auc = roc_auc_score(y_test, y_prob_final)

print(f"{'='*50}")
print(f"  FINAL MODEL: {best_model_name}")
print(f"{'='*50}")
print(f"  Test Accuracy  : {final_acc:.4f} ({final_acc*100:.2f}%)")
print(f"  Test ROC-AUC   : {final_auc:.4f}")
print(f"{'='*50}")
print()
print("📊 Classification Report:")
print(classification_report(y_test, y_pred_final, target_names=['Normal', 'Failure']))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('#0f0f0f')

# ── Confusion Matrix ──────────────────────────────────────────────────────────
ax1 = axes[0]
cm = confusion_matrix(y_test, y_pred_final)
cmap = sns.light_palette('#00d4ff', as_cmap=True)
sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax1,
            linewidths=2, linecolor='#0f0f0f',
            xticklabels=['Normal', 'Failure'],
            yticklabels=['Normal', 'Failure'],
            annot_kws={'size': 16, 'weight': 'bold', 'color': 'black'})
ax1.set_title('Confusion Matrix', fontsize=13, fontweight='bold', color='white', pad=12)
ax1.set_xlabel('Predicted', fontsize=11)
ax1.set_ylabel('Actual', fontsize=11)
tn, fp, fn, tp = cm.ravel()
ax1.text(0.5, -0.18, f'TN={tn}  FP={fp}  FN={fn}  TP={tp}',
         transform=ax1.transAxes, ha='center', fontsize=10, color='#aaa')

# ── ROC Curve ─────────────────────────────────────────────────────────────────
ax2 = axes[1]
fpr, tpr, _ = roc_curve(y_test, y_prob_final)
ax2.plot(fpr, tpr, color='#00d4ff', linewidth=2.5, label=f'AUC = {final_auc:.4f}')
ax2.fill_between(fpr, tpr, alpha=0.1, color='#00d4ff')
ax2.plot([0, 1], [0, 1], 'w--', linewidth=1, label='Random Classifier')
ax2.set_title('ROC Curve', fontsize=13, fontweight='bold', color='white', pad=12)
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.legend(fontsize=11, facecolor='#1a1a2e', edgecolor='white', labelcolor='white')
ax2.text(0.65, 0.1, f'AUC = {final_auc:.4f}', fontsize=14, fontweight='bold',
         color='#00d4ff', transform=ax2.transAxes)

# ── Feature Importance ────────────────────────────────────────────────────────
ax3 = axes[2]
importances = best_model.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values()
colors_bar = ['#ff4b6e' if v == feat_imp.max() else '#00d4ff' for v in feat_imp.values]
bars = ax3.barh(feat_imp.index, feat_imp.values, color=colors_bar,
                edgecolor='white', linewidth=0.5)
ax3.set_title('Feature Importance', fontsize=13, fontweight='bold', color='white', pad=12)
ax3.set_xlabel('Importance Score', fontsize=11)
for bar, val in zip(bars, feat_imp.values):
    ax3.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9, color='white')

plt.suptitle(f'Final Model Evaluation — {best_model_name}',
             fontsize=15, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', facecolor='#0f0f0f')
plt.show()


In [ ]:
# Cross-validation stability check on final model
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=10, scoring='roc_auc')

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0f0f0f')

ax.bar(range(1, 11), cv_scores, color='#a78bfa', edgecolor='white', linewidth=0.5, alpha=0.85)
ax.axhline(cv_scores.mean(), color='#00d4ff', linewidth=2, linestyle='--',
           label=f'Mean AUC = {cv_scores.mean():.4f}')
ax.fill_between(range(1, 11),
                cv_scores.mean() - cv_scores.std(),
                cv_scores.mean() + cv_scores.std(),
                alpha=0.15, color='#00d4ff', label=f'±1 Std ({cv_scores.std():.4f})')
ax.set_title('10-Fold Cross-Validation — ROC-AUC Stability', fontsize=13,
             fontweight='bold', color='white', pad=12)
ax.set_xlabel('Fold', fontsize=11)
ax.set_ylabel('ROC-AUC', fontsize=11)
ax.set_xticks(range(1, 11))
ax.set_ylim(max(0, cv_scores.min() - 0.05), 1.05)
ax.legend(fontsize=11, facecolor='#1a1a2e', edgecolor='white', labelcolor='white')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cv_stability.png', bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

print(f"10-Fold CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Min fold: {cv_scores.min():.4f}  |  Max fold: {cv_scores.max():.4f}")


## 8. Results Summary <a id='8-summary'></a>

In [ ]:
print("=" * 65)
print("           MACHINE FAILURE PREDICTION — RESULTS SUMMARY")
print("=" * 65)

print("\n📦 Dataset")
print(f"   Source     : Kaggle — Machine Failure Prediction using Sensor Data")
print(f"   Samples    : {len(data)} rows × {data.shape[1]} columns")
print(f"   Features   : {', '.join(X.columns)}")
print(f"   Target     : fail (0=Normal, 1=Failure)")

print("\n🔬 Models Evaluated")
for _, row in results_df.iterrows():
    flag = "🏆" if row['Model'] in best_model_name else "  "
    print(f"   {flag} {row['Model']:<22} Accuracy={row['Accuracy']:.4f}  AUC={row['ROC-AUC']:.4f}  F1(Fail)={row['F1-Failure']:.4f}")

print(f"\n🏆 Best Model : {best_model_name}")
print(f"   Test Accuracy  : {final_acc:.4f} ({final_acc*100:.2f}%)")
print(f"   Test ROC-AUC   : {final_auc:.4f}")
print(f"   10-Fold CV AUC : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

top_feat = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(f"\n📊 Top 3 Most Important Features")
for feat, imp in top_feat.head(3).items():
    print(f"   • {feat:<25} {imp:.4f}")

print("\n✅ Key Observations")
print("   • Sensor readings show clear distributional differences between")
print("     normal and failure states — confirming predictive signal in data.")
print("   • Model generalises well with stable cross-validation scores,")
print("     indicating low variance and no overfitting.")
print("   • Feature importance reveals which sensors most strongly signal")
print("     impending failure — actionable for maintenance teams.")
print("=" * 65)


---

## 📁 Artifacts Generated

| File | Description |
|---|---|
| `class_distribution.png` | Target class balance visualization |
| `sensor_histograms.png` | Feature distributions by failure class |
| `correlation_heatmap.png` | Feature correlation matrix |
| `boxplots.png` | Outlier analysis per feature |
| `pairplot.png` | Multi-feature interaction plot |
| `model_comparison.png` | Side-by-side model benchmarks |
| `feature_importance.png` | Final model feature importance + ROC curve |
| `cv_stability.png` | Cross-validation stability across folds |

---

## 🔗 References

- Dataset: [Machine Failure Prediction using Sensor Data — Kaggle](https://www.kaggle.com/datasets/umerrtx/machine-failure-prediction-using-sensor-data)
- scikit-learn: [https://scikit-learn.org](https://scikit-learn.org)
- XGBoost: [https://xgboost.readthedocs.io](https://xgboost.readthedocs.io)

---
*Kunduru Sneha | kundurusneha4@gmail.com | [GitHub: Sneh-04](https://github.com/Sneh-04)*
